### Carga dimensões (Dados Brutos)
**🔥 O que esse etapa faz?**
- ✅ Lê todos os arquivos CSV dentro de dimensao/.
- ✅ Transforma cada arquivo em Delta Parquet e salva na pasta correspondente (bronze/).
- ✅ Move os arquivos processados para processado/ para evitar reprocessamento.
- ✅ Mantém a flexibilidade para qualquer tabela sem precisar mudar o código.

Agora, os arquivos CSV serão automaticamente convertidos em Delta Tables e organizados corretamente na camada Bronze! 🚀

In [0]:
from pyspark.sql import SparkSession
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Configuração inicial da SparkSession com configurações otimizadas
spark = SparkSession.builder \
    .appName("Load Data Bronze") \
    .config("spark.sql.shuffle.partitions", "200")  \
    .config("spark.sql.files.maxPartitionBytes", "1GB") \
    .config("spark.sql.files.maxRecordsPerFile", "1000000") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

# Define um nome para a aplicação
# Define um número fixo de partições para shuffle, melhorando o paralelismo 
# Define o tamanho máximo de partições para evitar muitos arquivos pequenos 
# Define o tamanho máximo de linhas por arquivos para evitar muitos arquivos pequenos  
# Usa o codec Snappy para compressão rápida, otimizando tempo de leitura e escrita 
# Habilita otimizações adaptativas, ajustando o número de partições dinamicamente com base no tamanho dos dados   
# Obtém uma SparkSession ou, se não existir, cria uma baseada nas configs definidas   
# Inicializa a sessão do Spark (já está ativa no Databricks)


In [0]:
# Iniciando leitura dos arquivos CSV dentro de /dimensao 
# Transformando arquivos em Delta Parquet e 
# Salvando as dimensoes na pasta correspondente na camadada Bronze

# Diretórios Dimensao
dimensao_dir = "/mnt/panex/lhdw/landingzone/processar/dimensao/"
bronze_dir = "/mnt/panex/lhdw/bronze/dimensao/"
processado_dir = "/mnt/panex/lhdw/landingzone/processado/"

# Lista os arquivos CSV na pasta dimensao
arquivos = dbutils.fs.ls(dimensao_dir)

# Processa cada arquivo
for arquivo in arquivos:
    if arquivo.name.endswith(".csv"):
        nome_tabela = arquivo.name.replace(".csv", "")  # Exemplo: categoria.csv → categoria
        caminho_csv = arquivo.path
        caminho_delta = os.path.join(bronze_dir, nome_tabela)  # Exemplo: /mnt/panex/lhdw/bronze/categoria

        print(f"⏳Processando {arquivo.name}...")

        # Lê o CSV como DataFrame
        df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(caminho_csv)

        # Salva como Delta Parquet na pasta correspondente
        df.write.format("delta").mode("overwrite").save(caminho_delta)
        print(f"🟢 {arquivo.name} salvo como Delta em {caminho_delta}")

        # Move o arquivo processado para a pasta "processado"
        caminho_destino = os.path.join(processado_dir, arquivo.name)
        dbutils.fs.mv(caminho_csv, caminho_destino)
        print(f"➡️ {arquivo.name} movido para {caminho_destino}")

print("✅ Processamento concluído!")


⏳Processando categorias.csv...
🟢 categorias.csv salvo como Delta em /mnt/panex/lhdw/bronze/dimensao/categorias
➡️ categorias.csv movido para /mnt/panex/lhdw/landingzone/processado/categorias.csv
⏳Processando cidades.csv...
🟢 cidades.csv salvo como Delta em /mnt/panex/lhdw/bronze/dimensao/cidades
➡️ cidades.csv movido para /mnt/panex/lhdw/landingzone/processado/cidades.csv
⏳Processando clientes.csv...
🟢 clientes.csv salvo como Delta em /mnt/panex/lhdw/bronze/dimensao/clientes
➡️ clientes.csv movido para /mnt/panex/lhdw/landingzone/processado/clientes.csv
⏳Processando paises.csv...
🟢 paises.csv salvo como Delta em /mnt/panex/lhdw/bronze/dimensao/paises
➡️ paises.csv movido para /mnt/panex/lhdw/landingzone/processado/paises.csv
⏳Processando produtos.csv...
🟢 produtos.csv salvo como Delta em /mnt/panex/lhdw/bronze/dimensao/produtos
➡️ produtos.csv movido para /mnt/panex/lhdw/landingzone/processado/produtos.csv
⏳Processando vendedores.csv...
🟢 vendedores.csv salvo como Delta em /mnt/panex/l

In [0]:
# Evidencia de tabelas de dimensoes Delta-Parquet na camada Bronze
for folder in ["categorias", "produtos","cidades","clientes","vendedores","paises"]:
    print(f"\n✅Arquivos na pasta {folder}:")
    display(dbutils.fs.ls(f"/mnt/panex/lhdw/bronze/dimensao/{folder}/"))


✅Arquivos na pasta categorias:


path,name,size,modificationTime
dbfs:/mnt/panex/lhdw/bronze/dimensao/categorias/_delta_log/,_delta_log/,0,0
dbfs:/mnt/panex/lhdw/bronze/dimensao/categorias/part-00000-3959cdce-78b3-4b04-ada8-8c401e00b591-c000.snappy.parquet,part-00000-3959cdce-78b3-4b04-ada8-8c401e00b591-c000.snappy.parquet,1030,1741977606000



✅Arquivos na pasta produtos:


path,name,size,modificationTime
dbfs:/mnt/panex/lhdw/bronze/dimensao/produtos/_delta_log/,_delta_log/,0,0
dbfs:/mnt/panex/lhdw/bronze/dimensao/produtos/part-00000-9b259e83-c197-492c-9e3f-b58af86bd70c-c000.snappy.parquet,part-00000-9b259e83-c197-492c-9e3f-b58af86bd70c-c000.snappy.parquet,22465,1741977645000



✅Arquivos na pasta cidades:


...

path,name,size,modificationTime
dbfs:/mnt/panex/lhdw/bronze/dimensao/clientes/_delta_log/,_delta_log/,0,0
dbfs:/mnt/panex/lhdw/bronze/dimensao/clientes/part-00000-d8d32ff2-4b51-454e-a965-75eed156d3b6-c000.snappy.parquet,part-00000-d8d32ff2-4b51-454e-a965-75eed156d3b6-c000.snappy.parquet,1661146,1741977631000
dbfs:/mnt/panex/lhdw/bronze/dimensao/clientes/part-00001-575c0f38-e16b-46fc-826e-edc456bbf9d3-c000.snappy.parquet,part-00001-575c0f38-e16b-46fc-826e-edc456bbf9d3-c000.snappy.parquet,110918,1741977630000



✅Arquivos na pasta vendedores:


path,name,size,modificationTime
dbfs:/mnt/panex/lhdw/bronze/dimensao/vendedores/_delta_log/,_delta_log/,0,0
dbfs:/mnt/panex/lhdw/bronze/dimensao/vendedores/part-00000-6573da6d-977c-49e0-b00c-e7c28449132c-c000.snappy.parquet,part-00000-6573da6d-977c-49e0-b00c-e7c28449132c-c000.snappy.parquet,3443,1741977652000



✅Arquivos na pasta paises:


path,name,size,modificationTime
dbfs:/mnt/panex/lhdw/bronze/dimensao/paises/_delta_log/,_delta_log/,0,0
dbfs:/mnt/panex/lhdw/bronze/dimensao/paises/part-00000-abdaf781-b6b7-4b70-a993-c3f1cbdc1edb-c000.snappy.parquet,part-00000-abdaf781-b6b7-4b70-a993-c3f1cbdc1edb-c000.snappy.parquet,4688,1741977639000


#### Load Fato (Dados Brutos)
**🔥 O que esse etapa faz?**
- ✅Ler todos os arquivos dentro de /mnt/panex/lhdw/landingzone/processar/fato/.
- ✅Unir os arquivos em um único DataFrame, consolidando os dados de vendas.
- ✅Salvar como Delta Parquet na pasta /mnt/panex/lhdw/bronze/vendas/.
- ✅Mover os arquivos processados para /mnt/panex/lhdw/landingzone/processado/.

In [0]:
# Diretórios
fato_dir = "/mnt/panex/lhdw/landingzone/processar/fato/"
bronze_vendas_dir = "/mnt/panex/lhdw/bronze/fato"
processado_dir = "/mnt/panex/lhdw/landingzone/processado/"

# Lista os arquivos CSV na pasta fato
arquivos = dbutils.fs.ls(fato_dir)

# Lista para armazenar os DataFrames dos arquivos de fato
df_lista = []

# Processa cada arquivo
for arquivo in arquivos:
    if arquivo.name.endswith(".csv"):
        caminho_csv = arquivo.path
        print(f"Processando {arquivo.name}...")

        # Lê o CSV como DataFrame
        df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(caminho_csv)
        
        # Adiciona o DataFrame à lista
        df_lista.append(df)

# Se houver arquivos processados, consolidamos os dados corretamente
if df_lista:
    df_vendas = df_lista[0]
    for df in df_lista[1:]:  # Une os DataFrames iterativamente
        df_vendas = df_vendas.union(df)
    
    # Salva como Delta Parquet na pasta de vendas
    df_vendas.write.format("delta").mode("overwrite").save(bronze_vendas_dir)
    print(f"✅ Dados de vendas consolidados e salvos em {bronze_vendas_dir}")

    # Move os arquivos processados para a pasta "processado"
    for arquivo in arquivos:
        if arquivo.name.endswith(".csv"):
            caminho_csv = arquivo.path
            caminho_destino = os.path.join(processado_dir, arquivo.name)
            dbutils.fs.mv(caminho_csv, caminho_destino)
            print(f"✔ {arquivo.name} movido para {caminho_destino}")
else:
    print("⚠ Nenhum arquivo CSV encontrado na pasta fato.")

print("✅ Processamento concluído!")

Processando venda_1.csv...
Processando venda_2.csv...
Processando venda_3.csv...
✅ Dados de vendas consolidados e salvos em /mnt/panex/lhdw/bronze/fato
✔ venda_1.csv movido para /mnt/panex/lhdw/landingzone/processado/venda_1.csv
✔ venda_2.csv movido para /mnt/panex/lhdw/landingzone/processado/venda_2.csv
✔ venda_3.csv movido para /mnt/panex/lhdw/landingzone/processado/venda_3.csv
✅ Processamento concluído!


In [0]:
# Evidencia de tabela Fato Delta-Parquet na camada Bronze
display(dbutils.fs.ls(f"/mnt/panex/lhdw/bronze/fato"))

path,name,size,modificationTime
dbfs:/mnt/panex/lhdw/bronze/fato/_delta_log/,_delta_log/,0,0
dbfs:/mnt/panex/lhdw/bronze/fato/part-00000-26722afa-c7dc-4db3-a4dd-fde847d64b9c-c000.snappy.parquet,part-00000-26722afa-c7dc-4db3-a4dd-fde847d64b9c-c000.snappy.parquet,2272635,1741978247000
dbfs:/mnt/panex/lhdw/bronze/fato/part-00001-4aed94ca-6bd9-4c89-bfd6-a089555042a7-c000.snappy.parquet,part-00001-4aed94ca-6bd9-4c89-bfd6-a089555042a7-c000.snappy.parquet,2272902,1741978246000
dbfs:/mnt/panex/lhdw/bronze/fato/part-00002-e70128a7-805c-4972-9b28-f07f566a7372-c000.snappy.parquet,part-00002-e70128a7-805c-4972-9b28-f07f566a7372-c000.snappy.parquet,2272580,1741978246000
dbfs:/mnt/panex/lhdw/bronze/fato/part-00003-5e7695bf-9e6e-4745-9931-c39f44ff8a7a-c000.snappy.parquet,part-00003-5e7695bf-9e6e-4745-9931-c39f44ff8a7a-c000.snappy.parquet,2273227,1741978246000
dbfs:/mnt/panex/lhdw/bronze/fato/part-00004-d6911ce4-301f-4e6b-8529-4845b26f2d3c-c000.snappy.parquet,part-00004-d6911ce4-301f-4e6b-8529-4845b26f2d3c-c000.snappy.parquet,2272827,1741978246000
dbfs:/mnt/panex/lhdw/bronze/fato/part-00005-0181584b-943e-4b99-9604-41add62e79d1-c000.snappy.parquet,part-00005-0181584b-943e-4b99-9604-41add62e79d1-c000.snappy.parquet,2017181,1741978245000
dbfs:/mnt/panex/lhdw/bronze/fato/part-00006-9a1ee6ca-fb88-465e-8639-0f01d7c04d22-c000.snappy.parquet,part-00006-9a1ee6ca-fb88-465e-8639-0f01d7c04d22-c000.snappy.parquet,2273343,1741978247000
dbfs:/mnt/panex/lhdw/bronze/fato/part-00007-7d9c4264-03ac-44a9-b8c1-079684e6edb6-c000.snappy.parquet,part-00007-7d9c4264-03ac-44a9-b8c1-079684e6edb6-c000.snappy.parquet,2272819,1741978247000
dbfs:/mnt/panex/lhdw/bronze/fato/part-00008-ee29ac12-5eea-4a34-bff8-783971e437b7-c000.snappy.parquet,part-00008-ee29ac12-5eea-4a34-bff8-783971e437b7-c000.snappy.parquet,2272885,1741978251000
